# NUS ST5225 — Statistical Analysis of Networks
## Refactored OOP Case Study: SNAP Email-Eu-Core

This notebook is a deliberately **modular, object-oriented redesign** of the earlier ST5225 case study.

The statistical scope remains broad:

$$
\text{Describe}
\rightarrow
\text{Compare}
\rightarrow
\text{Model}
\rightarrow
\text{Infer}
\rightarrow
\text{Predict}
$$

but the implementation now behaves more like a small reusable analytics library.

### Real external dataset

We use Stanford SNAP's **Email-Eu-Core** network:

- 1,005 nodes,
- 25,571 directed email links,
- 42 department labels.

Primary source:

- https://snap.stanford.edu/data/email-Eu-core.html

Optional Kaggle mirror:

- https://www.kaggle.com/datasets/wolfram77/graphs-snap-email-eu

### ST5225 concepts covered

- directed and undirected networks,
- degree, density, components and clustering,
- reciprocity and assortativity,
- centrality and PageRank,
- adjacency matrices and graph Laplacians,
- spectral graph analysis,
- Erdős–Rényi and degree-preserving null models,
- community detection and hyperparameter tuning,
- stochastic block models,
- ERGM-style conditional modelling,
- structural and supervised link prediction,
- ranking evaluation,
- temporal-network extension,
- computational trade-offs.

### Software-design goals

The notebook demonstrates several design patterns without turning the analysis into unnecessary enterprise boilerplate:

1. **Repository pattern** — data acquisition and graph construction.
2. **Strategy pattern** — null models, community algorithms and dyadic features.
3. **Composite pattern** — combine many dyadic features through one extractor.
4. **Factory/Registry pattern** — model-search configuration.
5. **Template-style experiment runner** — common tuning/evaluation workflow.
6. **Facade pattern** — a compact `NetworkWorkbench` entry point.
7. **Dependency injection** — experiments receive strategies rather than hard-code algorithms.
8. **Caching/precomputation** — neighbourhood sets and degrees are computed once.

The important engineering principle is:

> Modularisation should reduce repetition and make experiments easier to extend, not merely create more classes.

## 1. Architecture

The notebook is organised into layers.

```text
┌────────────────────────────────────────────────────────────┐
│                    NetworkWorkbench                        │
│                       (Facade)                             │
└──────────────┬───────────────────────┬─────────────────────┘
               │                       │
      ┌────────▼────────┐     ┌────────▼─────────┐
      │ NetworkRepository│     │ Experiment Layer │
      │ data + graph     │     │ tuning/eval      │
      └────────┬────────┘     └────────┬─────────┘
               │                       │
      ┌────────▼────────┐     ┌────────▼─────────────┐
      │ NetworkAnalyzer │     │ Strategy Interfaces  │
      │ descriptive     │     │ null/community/link │
      └────────┬────────┘     └────────┬─────────────┘
               │                       │
      ┌────────▼────────┐     ┌────────▼─────────────┐
      │ SpectralAnalyzer│     │ CompositePairFeatures│
      │ matrices/eigens │     │ reusable dyad cache │
      └─────────────────┘     └──────────────────────┘
```

The benefit is that a new algorithm can often be added by implementing **one small strategy class** instead of rewriting an entire experiment.

## 2. Imports and configuration

`fast_mode=True` keeps expensive simulations and hyperparameter searches practical for teaching.

Switch it to `False` for a more exhaustive experiment.

In [1]:
# Optional installation if required:
# %pip install -q numpy pandas scipy networkx scikit-learn bokeh kagglehub

from __future__ import annotations

import math
import random
import shutil
import urllib.request
import warnings

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import (
    Any,
    Callable,
    Iterable,
    Mapping,
    Optional,
    Protocol,
    Sequence,
)

import numpy as np
import pandas as pd
import networkx as nx

from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.stats import spearmanr

import sklearn
from sklearn.base import BaseEstimator
from sklearn.cluster import SpectralClustering
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score,
    average_precision_score,
    f1_score,
    normalized_mutual_info_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import (
    BasicTicker,
    ColorBar,
    ColumnDataSource,
    HoverTool,
    LinearColorMapper,
)
from bokeh.palettes import Turbo256, Viridis256
from bokeh.plotting import figure, from_networkx

from IPython.display import display, Markdown

output_notebook(hide_banner=True)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

In [2]:
@dataclass(frozen=True)
class Config:
    data_dir: Path = Path("data_st5225")
    random_state: int = RANDOM_STATE
    fast_mode: bool = True

    # Null-model experiment
    null_simulations_fast: int = 8
    null_simulations_full: int = 40

    # Community experiments
    louvain_resolutions: tuple[float, ...] = (
        0.40, 0.60, 0.80, 1.00, 1.25, 1.50, 2.00, 2.50
    )
    spectral_k_fast: tuple[int, ...] = (8, 12, 20, 30, 42, 50)
    spectral_k_full: tuple[int, ...] = (
        5, 8, 10, 12, 15, 20, 25, 30, 35, 42, 50, 60
    )

    # Dyadic modelling
    ergm_positive_fast: int = 4_000
    ergm_positive_full: int = 12_000

    # Link prediction
    link_test_fraction: float = 0.15
    link_train_positive_fast: int = 5_000
    link_train_positive_full: int = 15_000
    negative_ratio: float = 1.0

    run_temporal_extension: bool = False

    @property
    def null_simulations(self) -> int:
        return self.null_simulations_fast if self.fast_mode else self.null_simulations_full

    @property
    def spectral_k_values(self) -> tuple[int, ...]:
        return self.spectral_k_fast if self.fast_mode else self.spectral_k_full

    @property
    def ergm_positive(self) -> int:
        return self.ergm_positive_fast if self.fast_mode else self.ergm_positive_full

    @property
    def link_train_positive(self) -> int:
        return (
            self.link_train_positive_fast
            if self.fast_mode
            else self.link_train_positive_full
        )


CFG = Config()
CFG.data_dir.mkdir(parents=True, exist_ok=True)
CFG

Config(data_dir=PosixPath('data_st5225'), random_state=42, fast_mode=True, null_simulations_fast=8, null_simulations_full=40, louvain_resolutions=(0.4, 0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5), spectral_k_fast=(8, 12, 20, 30, 42, 50), spectral_k_full=(5, 8, 10, 12, 15, 20, 25, 30, 35, 42, 50, 60), ergm_positive_fast=4000, ergm_positive_full=12000, link_test_fraction=0.15, link_train_positive_fast=5000, link_train_positive_full=15000, negative_ratio=1.0, run_temporal_extension=False)

# Part I — Data and graph repository

## 3. Repository pattern

The **Repository pattern** isolates data access from analysis.

The rest of the notebook should not care whether files came from:

- SNAP,
- Kaggle,
- a local cache,
- a database,
- an object store.

It simply asks the repository for a `NetworkDataset`.

This makes the analysis layer testable and reusable.

In [3]:
@dataclass
class NetworkDataset:
    edges: pd.DataFrame
    labels: pd.DataFrame
    directed: nx.DiGraph
    undirected: nx.Graph
    department_map: dict[int, int]

    @property
    def largest_connected_undirected(self) -> nx.Graph:
        nodes = max(nx.connected_components(self.undirected), key=len)
        return self.undirected.subgraph(nodes).copy()


class DataSource(Protocol):
    def load(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        ...


class SnapEmailEUDataSource:
    EDGE_FILE = "email-Eu-core.txt.gz"
    LABEL_FILE = "email-Eu-core-department-labels.txt.gz"
    SNAP_BASE = "https://snap.stanford.edu/data"
    KAGGLE_SLUG = "wolfram77/graphs-snap-email-eu"

    def __init__(self, data_dir: Path):
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def _download_from_snap(self, filename: str, target: Path) -> bool:
        try:
            urllib.request.urlretrieve(
                f"{self.SNAP_BASE}/{filename}",
                target,
            )
            return True
        except Exception as exc:
            print(f"SNAP download failed for {filename}: {exc}")
            return False

    def _download_from_kaggle(self, filename: str, target: Path) -> bool:
        try:
            import kagglehub
        except ImportError:
            return False

        try:
            root = Path(kagglehub.dataset_download(self.KAGGLE_SLUG))
            matches = list(root.rglob(filename))
            if not matches:
                return False
            shutil.copy2(matches[0], target)
            return True
        except Exception as exc:
            print(f"Kaggle fallback failed for {filename}: {exc}")
            return False

    def _ensure(self, filename: str) -> Path:
        target = self.data_dir / filename

        if target.exists():
            return target

        if self._download_from_snap(filename, target):
            return target

        if self._download_from_kaggle(filename, target):
            return target

        raise RuntimeError(
            f"Could not obtain {filename}. Download it manually from "
            "https://snap.stanford.edu/data/email-Eu-core.html and place it in "
            f"{self.data_dir.resolve()}."
        )

    def load(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        edge_path = self._ensure(self.EDGE_FILE)
        label_path = self._ensure(self.LABEL_FILE)

        edges = pd.read_csv(
            edge_path,
            sep=r"\s+",
            header=None,
            names=["src", "dst"],
            compression="gzip",
            dtype=int,
        )

        labels = pd.read_csv(
            label_path,
            sep=r"\s+",
            header=None,
            names=["node", "department"],
            compression="gzip",
            dtype=int,
        )

        return edges, labels


class NetworkRepository:
    def __init__(self, source: DataSource):
        self.source = source

    @staticmethod
    def audit(
        edges: pd.DataFrame,
        labels: pd.DataFrame,
    ) -> pd.DataFrame:
        edge_nodes = set(edges["src"]) | set(edges["dst"])
        label_nodes = set(labels["node"])

        rows = {
            "raw_edge_rows": len(edges),
            "duplicate_directed_edges": int(
                edges.duplicated(["src", "dst"]).sum()
            ),
            "self_loops": int((edges["src"] == edges["dst"]).sum()),
            "unique_edge_nodes": len(edge_nodes),
            "labelled_nodes": len(label_nodes),
            "nodes_missing_label": len(edge_nodes - label_nodes),
            "labels_without_edge_node": len(label_nodes - edge_nodes),
            "departments": int(labels["department"].nunique()),
        }

        return pd.DataFrame(
            {"metric": rows.keys(), "value": rows.values()}
        )

    def load(self) -> NetworkDataset:
        edges, labels = self.source.load()

        Gd = nx.from_pandas_edgelist(
            edges,
            source="src",
            target="dst",
            create_using=nx.DiGraph(),
        )
        Gd.add_nodes_from(labels["node"].tolist())

        department_map = dict(
            zip(labels["node"], labels["department"])
        )
        nx.set_node_attributes(Gd, department_map, "department")

        Gd.remove_edges_from(nx.selfloop_edges(Gd))

        Gu = Gd.to_undirected()
        Gu.remove_edges_from(nx.selfloop_edges(Gu))

        return NetworkDataset(
            edges=edges,
            labels=labels,
            directed=Gd,
            undirected=Gu,
            department_map=department_map,
        )

In [4]:
repository = NetworkRepository(
    SnapEmailEUDataSource(CFG.data_dir)
)

dataset = repository.load()

display(repository.audit(dataset.edges, dataset.labels))

Gd = dataset.directed
Gu = dataset.undirected
G_lcc = dataset.largest_connected_undirected
dept_map = dataset.department_map

print(f"Directed:   n={Gd.number_of_nodes():,}, m={Gd.number_of_edges():,}")
print(f"Undirected: n={Gu.number_of_nodes():,}, m={Gu.number_of_edges():,}")
print(f"LCC:        n={G_lcc.number_of_nodes():,}, m={G_lcc.number_of_edges():,}")

,metric,value
0,raw_edge_rows,25571
1,duplicate_directed_edges,0
2,self_loops,642
3,unique_edge_nodes,1005
4,labelled_nodes,1005
5,nodes_missing_label,0
6,labels_without_edge_node,0
7,departments,42


Directed:   n=1,005, m=24,929
Undirected: n=1,005, m=16,064
LCC:        n=986, m=16,064


## 4. Small plotting facade

Repeated Bokeh boilerplate obscures statistical ideas.

`PlotFactory` centralises common plot construction while leaving the data transformations visible.

This is intentionally a **thin facade**, not a huge visualisation framework.

In [5]:
class PlotFactory:
    @staticmethod
    def scatter(
        df: pd.DataFrame,
        x: str,
        y: str,
        title: str,
        x_label: str,
        y_label: str,
        hover: Optional[list[tuple[str, str]]] = None,
        width: int = 780,
        height: int = 430,
        x_axis_type: str = "linear",
        y_axis_type: str = "linear",
    ):
        p = figure(
            width=width,
            height=height,
            title=title,
            x_axis_label=x_label,
            y_axis_label=y_label,
            x_axis_type=x_axis_type,
            y_axis_type=y_axis_type,
        )

        source = ColumnDataSource(df)

        p.scatter(
            x=x,
            y=y,
            source=source,
            marker="circle",
            size=7,
            alpha=0.60,
        )

        if hover:
            p.add_tools(HoverTool(tooltips=hover))

        return p

    @staticmethod
    def multi_line(
        df: pd.DataFrame,
        x: str,
        series: Mapping[str, str],
        title: str,
        x_label: str,
        y_label: str,
        width: int = 780,
        height: int = 400,
        legend_location: str = "top_right",
    ):
        p = figure(
            width=width,
            height=height,
            title=title,
            x_axis_label=x_label,
            y_axis_label=y_label,
        )

        dash_patterns = ["solid", "dashed", "dotdash", "dotted"]

        for i, (column_name, legend_name) in enumerate(series.items()):
            p.line(
                df[x],
                df[column_name],
                line_width=3,
                line_dash=dash_patterns[i % len(dash_patterns)],
                legend_label=legend_name,
            )
            p.scatter(
                df[x],
                df[column_name],
                size=7,
                marker="circle",
            )

        if len(p.legend):
            p.legend.location = legend_location
            p.legend.click_policy = "hide"

        return p

    @staticmethod
    def heatmap(
        long_df: pd.DataFrame,
        x: str,
        y: str,
        value: str,
        x_values: Sequence[Any],
        y_values: Sequence[Any],
        title: str,
        palette=Viridis256,
        width: int = 730,
        height: int = 650,
    ):
        frame = long_df.copy()
        frame[x] = frame[x].astype(str)
        frame[y] = frame[y].astype(str)

        if frame.empty:
            raise ValueError("heatmap() received an empty DataFrame")

        value_min = float(frame[value].min())
        value_max = float(frame[value].max())
        if not np.isfinite(value_min) or not np.isfinite(value_max):
            raise ValueError(f"heatmap column {value!r} contains no finite values")
        if value_min == value_max:
            value_max = value_min + 1e-12

        mapper = LinearColorMapper(
            palette=palette,
            low=value_min,
            high=value_max,
        )

        p = figure(
            width=width,
            height=height,
            title=title,
            x_range=[str(v) for v in x_values],
            y_range=[str(v) for v in reversed(y_values)],
            tools="hover,save,reset",
            tooltips=[
                (x, f"@{x}"),
                (y, f"@{y}"),
                (value, f"@{value}{{0.00000}}"),
            ],
        )

        p.rect(
            x=x,
            y=y,
            width=1,
            height=1,
            source=ColumnDataSource(frame),
            fill_color={"field": value, "transform": mapper},
            line_color=None,
        )

        p.add_layout(
            ColorBar(
                color_mapper=mapper,
                ticker=BasicTicker(),
                label_standoff=8,
            ),
            "right",
        )

        return p

# Part II — Reusable descriptive analysis

## 5. `NetworkAnalyzer`

Instead of recomputing common graph summaries in unrelated cells, one analyzer owns the descriptive operations.

This class deliberately returns **DataFrames**, not plots.

That separation keeps:

$$
\text{analysis logic}
\neq
\text{presentation logic}.
$$

In [6]:
class NetworkAnalyzer:
    def __init__(self, dataset: NetworkDataset):
        self.data = dataset

    def global_summary(self) -> pd.DataFrame:
        Gd = self.data.directed
        Gu = self.data.undirected

        largest_wcc = max(nx.weakly_connected_components(Gd), key=len)
        largest_scc = max(nx.strongly_connected_components(Gd), key=len)

        metrics = {
            "nodes": Gd.number_of_nodes(),
            "directed_edges": Gd.number_of_edges(),
            "undirected_edges": Gu.number_of_edges(),
            "directed_density": nx.density(Gd),
            "undirected_density": nx.density(Gu),
            "reciprocity": nx.reciprocity(Gd),
            "weak_components": nx.number_weakly_connected_components(Gd),
            "strong_components": nx.number_strongly_connected_components(Gd),
            "largest_WCC_nodes": len(largest_wcc),
            "largest_SCC_nodes": len(largest_scc),
            "average_clustering": nx.average_clustering(Gu),
            "transitivity": nx.transitivity(Gu),
            "department_assortativity": nx.attribute_assortativity_coefficient(
                Gu,
                "department",
            ),
        }

        return pd.DataFrame(
            {"metric": metrics.keys(), "value": metrics.values()}
        )

    def degree_table(self) -> pd.DataFrame:
        Gd = self.data.directed
        Gu = self.data.undirected

        return pd.DataFrame(
            {
                "node": list(Gd.nodes()),
                "in_degree": [Gd.in_degree(v) for v in Gd.nodes()],
                "out_degree": [Gd.out_degree(v) for v in Gd.nodes()],
                "undirected_degree": [Gu.degree(v) for v in Gd.nodes()],
                "department": [
                    self.data.department_map[v]
                    for v in Gd.nodes()
                ],
            }
        )

    @staticmethod
    def empirical_distribution(
        values: Iterable[int],
        value_name: str = "degree",
    ) -> pd.DataFrame:
        s = pd.Series(list(values), name=value_name)

        out = (
            s.value_counts()
            .sort_index()
            .rename("count")
            .reset_index()
        )

        out["probability"] = out["count"] / out["count"].sum()
        return out

    def centrality_table(self) -> pd.DataFrame:
        G = self.data.largest_connected_undirected

        degree = nx.degree_centrality(G)
        closeness = nx.closeness_centrality(G)
        betweenness = nx.betweenness_centrality(G, normalized=True)

        try:
            eigenvector = nx.eigenvector_centrality(
                G,
                max_iter=1_000,
            )
        except nx.PowerIterationFailedConvergence:
            eigenvector = nx.eigenvector_centrality_numpy(G)

        pagerank = nx.pagerank(
            self.data.directed,
            alpha=0.85,
        )

        nodes = sorted(G.nodes())

        return pd.DataFrame(
            {
                "node": nodes,
                "degree": [degree[v] for v in nodes],
                "closeness": [closeness[v] for v in nodes],
                "betweenness": [betweenness[v] for v in nodes],
                "eigenvector": [eigenvector[v] for v in nodes],
                "pagerank": [pagerank[v] for v in nodes],
                "department": [
                    self.data.department_map[v]
                    for v in nodes
                ],
            }
        )

    def department_mixing(
        self,
        top_n: int = 15,
    ) -> tuple[pd.DataFrame, list[int]]:
        Gu = self.data.undirected
        dept_map = self.data.department_map

        dept_sizes = self.data.labels["department"].value_counts()
        top_departments = dept_sizes.head(top_n).index.tolist()

        rows = []

        for u, v in Gu.edges():
            du = dept_map[u]
            dv = dept_map[v]

            if du not in top_departments or dv not in top_departments:
                continue

            rows.append((du, dv))

            if du != dv:
                rows.append((dv, du))

        mixing = (
            pd.DataFrame(rows, columns=["dept_a", "dept_b"])
            .groupby(["dept_a", "dept_b"])
            .size()
            .rename("edges")
            .reset_index()
        )

        all_pairs = pd.MultiIndex.from_product(
            [top_departments, top_departments],
            names=["dept_a", "dept_b"],
        ).to_frame(index=False)

        mixing = (
            all_pairs.merge(
                mixing,
                on=["dept_a", "dept_b"],
                how="left",
            )
            .fillna({"edges": 0})
        )

        return mixing, top_departments

In [7]:
analyzer = NetworkAnalyzer(dataset)

display(analyzer.global_summary().round(5))

degree_df = analyzer.degree_table()
display(
    degree_df.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(3)
)

,metric,value
0,nodes,1005.00000
1,directed_edges,24929.00000
2,undirected_edges,16064.00000
3,directed_density,0.02471
4,undirected_density,0.03184
5,reciprocity,0.71122
6,weak_components,20.00000
7,strong_components,203.00000
8,largest_WCC_nodes,986.00000
9,largest_SCC_nodes,803.00000


,node,in_degree,out_degree,undirected_degree,department
count,1005.000,1005.000,1005.000,1005.000,1005.000
mean,502.000,24.805,24.805,31.968,13.987
std,290.263,27.784,33.140,36.960,10.357
min,0.000,0.000,0.000,0.000,0.000
50%,502.000,17.000,14.000,21.000,14.000
75%,753.000,35.000,34.000,44.000,21.000
90%,903.600,60.000,63.600,76.200,31.000
95%,953.800,81.800,88.000,104.800,36.000
99%,993.960,126.960,154.560,168.960,38.000
max,1004.000,211.000,333.000,345.000,41.000


### Degree distribution

For an undirected graph,

$$
d_i=\sum_jA_{ij}.
$$

The degree distribution

$$
P(D=k)
$$

reveals heterogeneity that the mean degree alone hides.

In [43]:
degree_dist = analyzer.empirical_distribution(
    degree_df["undirected_degree"],
    value_name="undirected_degree"
)

show(
    PlotFactory.scatter(
        degree_dist,
        x="undirected_degree",
        y="probability",
        title="Empirical undirected degree distribution",
        x_label="Degree k",
        y_label="P(D = k)",
        hover=[
            ("degree", "@undirected_degree"),
            ("count", "@count"),
            ("probability", "@probability{0.0000}"),
        ],
        x_axis_type="log",
        y_axis_type="log",
    )
)

### Centrality

Different centralities encode different notions of importance:

$$
C_D(i)=d_i
$$

for immediate popularity,

$$
C_C(i)
=
\frac{n-1}{\sum_{j\neq i}d(i,j)}
$$

for closeness,

and

$$
C_B(v)
=
\sum_{s\neq v\neq t}
\frac{\sigma_{st}(v)}{\sigma_{st}}
$$

for brokerage.

Eigenvector centrality and PageRank incorporate the importance of neighbouring nodes.

In [9]:
centrality_df = analyzer.centrality_table()

display(
    centrality_df
    .sort_values("betweenness", ascending=False)
    .head(15)
    .round(6)
)

show(
    PlotFactory.scatter(
        centrality_df,
        x="degree",
        y="betweenness",
        title="Degree centrality vs betweenness",
        x_label="Degree centrality",
        y_label="Betweenness centrality",
        hover=[
            ("node", "@node"),
            ("department", "@department"),
            ("degree", "@degree{0.0000}"),
            ("betweenness", "@betweenness{0.0000}"),
        ],
    )
)

,node,degree,closeness,betweenness,eigenvector,pagerank,department
160,160,0.350254,0.584917,0.090821,0.165683,0.007495,36
86,86,0.219289,0.532145,0.039261,0.112104,0.005711,36
5,5,0.171574,0.511157,0.032203,0.079302,0.005122,25
82,82,0.234518,0.544500,0.028967,0.145129,0.003860,36
121,121,0.235533,0.541506,0.028927,0.148288,0.005232,36
107,107,0.222335,0.533875,0.025289,0.139729,0.005568,36
13,13,0.180711,0.512754,0.024483,0.085529,0.002399,26
377,377,0.139086,0.474928,0.024078,0.052432,0.003761,7
62,62,0.217259,0.532720,0.023387,0.131356,0.005897,36
64,64,0.170558,0.516518,0.022778,0.089781,0.004677,25


## 6. Generic parameter-sweep helper

PageRank damping, Louvain resolution and other experiments all follow the same pattern:

1. choose parameter values,
2. run a callable,
3. collect dictionaries,
4. return a tidy DataFrame.

`ParameterSweep` removes that repetition.

In [10]:
class ParameterSweep:
    @staticmethod
    def run(
        parameter_name: str,
        values: Iterable[Any],
        evaluator: Callable[[Any], Mapping[str, Any]],
    ) -> pd.DataFrame:
        rows = []

        for value in values:
            rows.append(
                {
                    parameter_name: value,
                    **dict(evaluator(value)),
                }
            )

        return pd.DataFrame(rows)

### PageRank damping-factor sensitivity

PageRank can be written schematically as

$$
\pi^\top
=
\pi^\top
\left[
\alpha P
+
(1-\alpha)\frac{\mathbf 1\mathbf 1^\top}{n}
\right].
$$

We compare rankings at different \(\alpha\) values with the conventional \(\alpha=0.85\) reference.

In [42]:
alphas = [0.65, 0.75, 0.85, 0.90, 0.95]

pageranks = {
    alpha: nx.pagerank(Gd, alpha=alpha, max_iter=500)
    for alpha in alphas
}

nodes_sorted = sorted(Gd.nodes())
reference_alpha = 0.85
reference_scores = np.array(
    [pageranks[reference_alpha][v] for v in nodes_sorted]
)

reference_top20 = set(
    sorted(
        pageranks[reference_alpha],
        key=pageranks[reference_alpha].get,
        reverse=True,
    )[:20]
)

def evaluate_pagerank_alpha(alpha: float) -> dict[str, float]:
    scores = np.array(
        [pageranks[alpha][v] for v in nodes_sorted]
    )

    top20 = set(
        sorted(
            pageranks[alpha],
            key=pageranks[alpha].get,
            reverse=True,
        )[:20]
    )

    return {
        "spearman_vs_0.85": spearmanr(
            reference_scores,
            scores,
        ).statistic,
        "top20_overlap": len(reference_top20 & top20) / 20,
    }


pagerank_tuning = ParameterSweep.run(
    "alpha",
    alphas,
    evaluate_pagerank_alpha,
)

display(pagerank_tuning.round(4))

show(
    PlotFactory.multi_line(
        pagerank_tuning,
        x="alpha",
        series={
            "spearman_vs_0.85": "Spearman vs α=0.85",
            "top20_overlap": "Top-20 overlap",
        },
        title="PageRank damping-factor sensitivity",
        x_label="α",
        y_label="Ranking stability",
    )
)

,alpha,spearman_vs_0.85,top20_overlap
0,0.65,0.9970,0.95
1,0.75,0.9991,1.00
2,0.85,1.0000,1.00
3,0.90,0.9997,1.00
4,0.95,0.9988,0.95


# Part III — Matrix and spectral layer

## 7. `SpectralAnalyzer`

Matrix construction and eigen-analysis are kept together.

For

$$
L=D-A,
$$

the quadratic form

$$
x^\top Lx
=
\frac12\sum_{i,j}A_{ij}(x_i-x_j)^2
$$

measures lack of smoothness over edges.

The second eigenvector of \(L\), the **Fiedler vector**, provides a relaxed graph partition.

In [12]:
class SpectralAnalyzer:
    def __init__(self, graph: nx.Graph):
        if not nx.is_connected(graph):
            raise ValueError(
                "SpectralAnalyzer expects a connected graph."
            )

        self.graph = graph
        self.nodes = sorted(graph.nodes())

        self.adjacency = sparse.csr_matrix(
            nx.to_scipy_sparse_array(
                graph,
                nodelist=self.nodes,
                dtype=float,
                format="csr",
            )
        )

        self.laplacian = sparse.csr_matrix(
            nx.laplacian_matrix(
                graph,
                nodelist=self.nodes,
            ).astype(float)
        )

    def smallest_eigenpairs(
        self,
        k: int = 10,
    ) -> tuple[np.ndarray, np.ndarray]:
        n = self.laplacian.shape[0]
        if n < 2:
            raise ValueError("At least two nodes are required for spectral analysis.")
        k = min(int(k), n - 1)
        if k < 1:
            raise ValueError("k must be at least 1.")

        values, vectors = eigsh(
            self.laplacian,
            k=k,
            which="SM",
        )

        order = np.argsort(values)
        return values[order], vectors[:, order]

    def fiedler_partition(self) -> pd.DataFrame:
        values, vectors = self.smallest_eigenpairs(k=3)
        fiedler = vectors[:, 1]

        return pd.DataFrame(
            {
                "node": self.nodes,
                "fiedler": fiedler,
                "spectral_side": (fiedler >= 0).astype(int),
            }
        )


spectral = SpectralAnalyzer(G_lcc)

eigvals, eigvecs = spectral.smallest_eigenpairs(k=10)

eigen_df = pd.DataFrame(
    {
        "index": np.arange(1, len(eigvals) + 1),
        "eigenvalue": eigvals,
    }
)

display(eigen_df.round(8))

show(
    PlotFactory.multi_line(
        eigen_df,
        x="index",
        series={"eigenvalue": "λ"},
        title="Smallest Laplacian eigenvalues",
        x_label="Eigenvalue index",
        y_label="λ",
    )
)

fiedler_df = spectral.fiedler_partition()
display(fiedler_df.head())
print(
    "Fiedler split:",
    fiedler_df["spectral_side"].value_counts().to_dict(),
)

,index,eigenvalue
0,1,0.000000
1,2,0.564121
2,3,0.693858
3,4,0.706246
4,5,0.857513
5,6,0.875979
6,7,0.896614
7,8,0.899408
8,9,0.909073
9,10,0.913407


,node,fiedler,spectral_side
0,0,-0.002238,0
1,1,-0.002366,0
2,2,-0.002592,0
3,3,-0.002555,0
4,4,-0.002595,0


Fiedler split: {0: 966, 1: 20}


## 8. Department mixing and homophily

Categorical assortativity measures the tendency of edges to connect nodes with the same category.

A positive department assortativity coefficient suggests that same-department communication is more common than expected under a suitable mixing baseline.

In [13]:
mixing_df, top_departments = analyzer.department_mixing(top_n=15)

show(
    PlotFactory.heatmap(
        mixing_df,
        x="dept_a",
        y="dept_b",
        value="edges",
        x_values=top_departments,
        y_values=top_departments,
        title="Department mixing — 15 largest departments",
    )
)

# Part IV — Null-model experiment framework

## 9. Strategy pattern for random graph models

A **Strategy** encapsulates one interchangeable algorithm behind a common interface.

Here:

```text
NullModelStrategy
├── ErdosRenyiStrategy
└── DegreePreservingStrategy
```

The experiment runner does not care how a strategy creates its graph.

That is useful because we can later add:

- Chung–Lu,
- configuration-model variants,
- SBM-based nulls,
- spatial null models,

without changing the experiment runner.

In [14]:
class NullModelStrategy(ABC):
    name: str

    @abstractmethod
    def sample(
        self,
        observed: nx.Graph,
        seed: int,
    ) -> nx.Graph:
        ...


class ErdosRenyiStrategy(NullModelStrategy):
    name = "Erdos-Renyi"

    def sample(
        self,
        observed: nx.Graph,
        seed: int,
    ) -> nx.Graph:
        return nx.gnp_random_graph(
            observed.number_of_nodes(),
            nx.density(observed),
            seed=seed,
        )


class DegreePreservingStrategy(NullModelStrategy):
    name = "Degree-preserving"

    def __init__(
        self,
        swaps_per_edge: int = 5,
        max_swaps: int = 80_000,
    ):
        self.swaps_per_edge = swaps_per_edge
        self.max_swaps = max_swaps

    def sample(
        self,
        observed: nx.Graph,
        seed: int,
    ) -> nx.Graph:
        graph = observed.copy()

        nswap = min(
            self.swaps_per_edge * graph.number_of_edges(),
            self.max_swaps,
        )

        try:
            nx.double_edge_swap(
                graph,
                nswap=nswap,
                max_tries=max(20 * nswap, 1_000),
                seed=seed,
            )
        except nx.NetworkXAlgorithmError:
            # A partially rewired graph is still a useful sample.
            pass

        return graph


class NullModelExperiment:
    METRICS = (
        "avg_clustering",
        "transitivity",
        "triangles",
    )

    def __init__(
        self,
        graph: nx.Graph,
        strategies: Sequence[NullModelStrategy],
        random_state: int = 42,
    ):
        self.graph = graph
        self.strategies = list(strategies)
        self.random_state = random_state

    @staticmethod
    def summarize(graph: nx.Graph) -> dict[str, float]:
        return {
            "edges": graph.number_of_edges(),
            "avg_clustering": nx.average_clustering(graph),
            "transitivity": nx.transitivity(graph),
            "triangles": sum(nx.triangles(graph).values()) // 3,
        }

    def run(
        self,
        n_simulations: int,
    ) -> pd.DataFrame:
        rows = []

        for strategy in self.strategies:
            for i in range(n_simulations):
                sampled = strategy.sample(
                    self.graph,
                    self.random_state + i,
                )

                rows.append(
                    {
                        "model": strategy.name,
                        "simulation": i,
                        **self.summarize(sampled),
                    }
                )

        return pd.DataFrame(rows)

    def z_score_table(
        self,
        results: pd.DataFrame,
    ) -> pd.DataFrame:
        observed = self.summarize(self.graph)
        rows = []

        for model_name, group in results.groupby("model"):
            for metric in self.METRICS:
                sd = group[metric].std(ddof=1)

                rows.append(
                    {
                        "model": model_name,
                        "metric": metric,
                        "observed": observed[metric],
                        "null_mean": group[metric].mean(),
                        "null_sd": sd,
                        "z_score": (
                            (observed[metric] - group[metric].mean()) / sd
                            if sd > 0
                            else np.nan
                        ),
                    }
                )

        return pd.DataFrame(rows)

In [15]:
null_experiment = NullModelExperiment(
    G_lcc,
    strategies=[
        ErdosRenyiStrategy(),
        DegreePreservingStrategy(),
    ],
    random_state=CFG.random_state,
)

null_results = null_experiment.run(
    CFG.null_simulations
)

display(
    null_results
    .groupby("model")[
        ["avg_clustering", "transitivity", "triangles"]
    ]
    .agg(["mean", "std"])
    .round(4)
)

display(
    null_experiment
    .z_score_table(null_results)
    .round(4)
)

avg_clustering         transitivity         triangles  \
                            mean     std         mean     std      mean   
model                                                                     
Degree-preserving         0.1620  0.0030       0.1433  0.0018  56537.75   
Erdos-Renyi               0.0332  0.0008       0.0331  0.0007   5717.25   

                             
                        std  
model                        
Degree-preserving  717.7962  
Erdos-Renyi        193.1504

,model,metric,observed,null_mean,null_sd,z_score
0,Degree-preserving,avg_clustering,0.4071,0.1620,0.0030,81.2899
1,Degree-preserving,transitivity,0.2674,0.1433,0.0018,68.1576
2,Degree-preserving,triangles,105461.0000,56537.7500,717.7962,68.1576
3,Erdos-Renyi,avg_clustering,0.4071,0.0332,0.0008,479.4774
4,Erdos-Renyi,transitivity,0.2674,0.0331,0.0007,327.5772
5,Erdos-Renyi,triangles,105461.0000,5717.2500,193.1504,516.4046


### Statistical interpretation

The nulls answer different questions.

#### Erdős–Rényi

Controls mainly for

$$
n
\quad\text{and}\quad
p.
$$

#### Degree-preserving rewiring

Controls for the complete degree sequence

$$
d_1,\ldots,d_n.
$$

If observed clustering remains extreme relative to the second null, degree heterogeneity alone is insufficient.

# Part V — Community detection as interchangeable strategies

## 10. Community strategy interface

We use the same interface for Louvain and spectral clustering.

This lets one generic evaluator calculate:

- number of communities,
- modularity,
- ARI,
- NMI.

The experiment code no longer duplicates those metrics for every algorithm.

In [16]:
class CommunityStrategy(ABC):
    name: str
    parameter_name: str

    @abstractmethod
    def detect(
        self,
        graph: nx.Graph,
        parameter: Any,
        random_state: int,
    ) -> list[set[int]]:
        ...


class LouvainCommunityStrategy(CommunityStrategy):
    name = "Louvain"
    parameter_name = "resolution"

    def detect(
        self,
        graph: nx.Graph,
        parameter: float,
        random_state: int,
    ) -> list[set[int]]:
        if hasattr(nx.community, "louvain_communities"):
            return [
                set(c)
                for c in nx.community.louvain_communities(
                    graph,
                    resolution=float(parameter),
                    seed=random_state,
                )
            ]

        return [
            set(c)
            for c in nx.community.greedy_modularity_communities(graph)
        ]


class SpectralCommunityStrategy(CommunityStrategy):
    name = "Spectral"
    parameter_name = "k"

    def detect(
        self,
        graph: nx.Graph,
        parameter: int,
        random_state: int,
    ) -> list[set[int]]:
        nodes = np.array(sorted(graph.nodes()))
        k = int(parameter)
        if not 2 <= k < len(nodes):
            raise ValueError(
                f"Spectral clustering requires 2 <= k < n_nodes; got k={k}, n_nodes={len(nodes)}"
            )

        adjacency = sparse.csr_matrix(
            nx.to_scipy_sparse_array(
                graph,
                nodelist=nodes.tolist(),
                dtype=float,
                format="csr",
            )
        )

        model = SpectralClustering(
            n_clusters=k,
            affinity="precomputed",
            assign_labels="kmeans",
            n_init=10,
            eigen_solver="arpack",
            random_state=random_state,
        )

        labels = model.fit_predict(adjacency)

        return [
            set(nodes[labels == label].tolist())
            for label in np.unique(labels)
        ]


class CommunityExperiment:
    def __init__(
        self,
        graph: nx.Graph,
        ground_truth: Mapping[int, int],
        random_state: int = 42,
    ):
        self.graph = graph
        self.ground_truth = ground_truth
        self.random_state = random_state
        self.nodes = sorted(graph.nodes())
        self.y_true = np.array(
            [ground_truth[v] for v in self.nodes]
        )

    def communities_to_labels(
        self,
        communities: Sequence[set[int]],
    ) -> np.ndarray:
        mapping = {}

        for label, community in enumerate(communities):
            for node in community:
                mapping[node] = label

        return np.array(
            [mapping[node] for node in self.nodes]
        )

    def evaluate(
        self,
        strategy: CommunityStrategy,
        parameter: Any,
    ) -> dict[str, Any]:
        communities = strategy.detect(
            self.graph,
            parameter,
            self.random_state,
        )

        y_pred = self.communities_to_labels(communities)

        return {
            "algorithm": strategy.name,
            "n_communities": len(communities),
            "modularity": nx.community.modularity(
                self.graph,
                communities,
            ),
            "ARI": adjusted_rand_score(
                self.y_true,
                y_pred,
            ),
            "NMI": normalized_mutual_info_score(
                self.y_true,
                y_pred,
            ),
        }

    def sweep(
        self,
        strategy: CommunityStrategy,
        parameters: Iterable[Any],
    ) -> pd.DataFrame:
        return ParameterSweep.run(
            strategy.parameter_name,
            parameters,
            lambda value: self.evaluate(strategy, value),
        )

In [17]:
community_experiment = CommunityExperiment(
    G_lcc,
    ground_truth=dept_map,
    random_state=CFG.random_state,
)

louvain_strategy = LouvainCommunityStrategy()

louvain_results = community_experiment.sweep(
    louvain_strategy,
    CFG.louvain_resolutions,
)

display(louvain_results.round(4))

show(
    PlotFactory.multi_line(
        louvain_results,
        x="resolution",
        series={
            "ARI": "ARI",
            "NMI": "NMI",
            "modularity": "Modularity",
        },
        title="Louvain resolution tuning",
        x_label="Resolution",
        y_label="Score",
    )
)

,resolution,algorithm,n_communities,modularity,ARI,NMI
0,0.40,Louvain,2,0.1043,0.0159,0.1474
1,0.60,Louvain,4,0.3457,0.1024,0.3974
2,0.80,Louvain,6,0.4053,0.2133,0.5164
3,1.00,Louvain,8,0.4111,0.3581,0.5919
4,1.25,Louvain,12,0.4051,0.4716,0.6466
5,1.50,Louvain,15,0.3982,0.4972,0.6737
6,2.00,Louvain,23,0.3801,0.5512,0.7070
7,2.50,Louvain,31,0.3744,0.5585,0.7184


In [18]:
spectral_strategy = SpectralCommunityStrategy()

spectral_results = community_experiment.sweep(
    spectral_strategy,
    CFG.spectral_k_values,
)

display(spectral_results.round(4))

show(
    PlotFactory.multi_line(
        spectral_results,
        x="k",
        series={
            "ARI": "ARI",
            "NMI": "NMI",
            "modularity": "Modularity",
        },
        title="Spectral clustering — sensitivity to K",
        x_label="K",
        y_label="Score",
    )
)

,k,algorithm,n_communities,modularity,ARI,NMI
0,8,Spectral,8,0.3247,0.1048,0.4429
1,12,Spectral,12,0.3020,0.1097,0.4682
2,20,Spectral,20,0.3094,0.1879,0.5600
3,30,Spectral,30,0.2626,0.1495,0.5659
4,42,Spectral,42,0.2080,0.0907,0.5091
5,50,Spectral,50,0.1494,0.0639,0.4803


### Why ARI/NMI and modularity need not agree

The formal department labels answer:

> Which organisational unit does this person belong to?

Graph communities answer:

> Which nodes are densely connected relative to a network baseline?

Those are related but not identical questions.

A department can split into several communication communities, or several departments can form one operational cluster.

# Part VI — Empirical stochastic block model

## 11. `EmpiricalSBM`

For known group labels,

$$
Y_{ij}\mid z_i=a,z_j=b
\sim
\operatorname{Bernoulli}(B_{ab}).
$$

The MLE is the observed block density:

$$
\hat B_{ab}
=
\frac{m_{ab}}{N_{ab}}.
$$

The implementation below packages fitting and prediction behind a reusable class.

In [19]:
class EmpiricalSBM:
    def __init__(self, smoothing: float = 1.0):
        self.smoothing = float(smoothing)
        self.groups_: Optional[list[int]] = None
        self.group_index_: Optional[dict[int, int]] = None
        self.B_: Optional[np.ndarray] = None

    def fit(
        self,
        graph: nx.Graph,
        group_map: Mapping[int, int],
    ) -> "EmpiricalSBM":
        groups = sorted(
            {group_map[v] for v in graph.nodes()}
        )

        group_index = {
            group: i
            for i, group in enumerate(groups)
        }

        nodes_by_group = {
            group: [
                node
                for node in graph.nodes()
                if group_map[node] == group
            ]
            for group in groups
        }

        K = len(groups)
        edge_count = np.zeros((K, K), dtype=float)
        possible = np.zeros((K, K), dtype=float)

        for a, group_a in enumerate(groups):
            n_a = len(nodes_by_group[group_a])

            for b, group_b in enumerate(groups):
                n_b = len(nodes_by_group[group_b])

                possible[a, b] = (
                    n_a * (n_a - 1) / 2
                    if a == b
                    else n_a * n_b
                )

        for u, v in graph.edges():
            a = group_index[group_map[u]]
            b = group_index[group_map[v]]

            edge_count[a, b] += 1

            if a != b:
                edge_count[b, a] += 1

        B = (
            edge_count + self.smoothing
        ) / (
            possible + 2 * self.smoothing
        )

        self.groups_ = groups
        self.group_index_ = group_index
        self.B_ = B

        return self

    def probability(
        self,
        u: int,
        v: int,
        group_map: Mapping[int, int],
    ) -> float:
        if self.B_ is None or self.group_index_ is None:
            raise RuntimeError("Call fit() before probability().")

        a = self.group_index_[group_map[u]]
        b = self.group_index_[group_map[v]]

        return float(self.B_[a, b])

    def to_frame(self) -> pd.DataFrame:
        if self.B_ is None or self.groups_ is None:
            raise RuntimeError("Call fit() before to_frame().")

        return pd.DataFrame(
            self.B_,
            index=self.groups_,
            columns=self.groups_,
        )


sbm = EmpiricalSBM(smoothing=1.0).fit(
    G_lcc,
    dept_map,
)

B_df = sbm.to_frame()

long_B = (
    B_df
    .stack()
    .rename("edge_probability")
    .reset_index()
    .rename(
        columns={
            "level_0": "department_a",
            "level_1": "department_b",
        }
    )
)

display(
    long_B
    .sort_values("edge_probability", ascending=False)
    .head(25)
    .round(5)
)

,department_a,department_b,edge_probability
1075,25,25,0.94118
1720,40,40,0.87500
516,12,12,0.80000
215,5,5,0.60000
1591,37,37,0.57944
1548,36,36,0.53219
86,2,2,0.53191
688,16,16,0.52878
1118,26,26,0.52632
473,11,11,0.50980


In [20]:
largest_departments = (
    dataset.labels["department"]
    .value_counts()
    .head(15)
    .index
    .tolist()
)

sbm_heat = long_B[
    long_B["department_a"].isin(largest_departments)
    & long_B["department_b"].isin(largest_departments)
]

show(
    PlotFactory.heatmap(
        sbm_heat,
        x="department_a",
        y="department_b",
        value="edge_probability",
        x_values=largest_departments,
        y_values=largest_departments,
        title="Empirical SBM edge probabilities",
        palette=Turbo256,
    )
)

# Part VII — Dyadic feature framework

## 12. Strategy + Composite patterns

ERGM-style modelling and link prediction need many of the **same pairwise graph features**.

The earlier notebook computed neighbourhood sets separately in several places.

This version introduces:

```text
PairContext
    ├── cached neighbours
    ├── cached degree
    └── department map

PairFeatureStrategy
    ├── CommonNeighbors
    ├── Jaccard
    ├── AdamicAdar
    ├── ResourceAllocation
    ├── PreferentialAttachment
    ├── SameDepartment
    └── Reciprocity

CompositePairFeatureExtractor
    └── combines any subset of strategies
```

This is both cleaner and more efficient.

If one feature calculation is \(O(d_u+d_v)\), recalculating neighbourhood sets across many feature functions wastes work.

The cached context performs that setup once.

In [41]:
@dataclass
class PairContext:
    undirected: nx.Graph
    department_map: Mapping[int, int]
    directed: Optional[nx.DiGraph] = None

    neighbours: dict[int, frozenset[int]] = field(init=False)
    degree: dict[int, int] = field(init=False)

    def __post_init__(self):
        self.neighbours = {
            node: frozenset(self.undirected.neighbors(node))
            for node in self.undirected.nodes()
        }

        self.degree = {
            node: len(self.neighbours[node])
            for node in self.undirected.nodes()
        }

    def neighbour_pair(
        self,
        u: int,
        v: int,
        exclude_dyad: bool = True,
    ) -> tuple[set[int], set[int]]:
        Nu = set(self.neighbours[u])
        Nv = set(self.neighbours[v])

        if exclude_dyad:
            Nu.discard(v)
            Nv.discard(u)

        return Nu, Nv


class PairFeatureStrategy(ABC):
    name: str

    @abstractmethod
    def compute(
        self,
        context: PairContext,
        u: int,
        v: int,
    ) -> float:
        ...


class CommonNeighbors(PairFeatureStrategy):
    name = "common_neighbors"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)
        return float(len(Nu & Nv))


class JaccardFeature(PairFeatureStrategy):
    name = "jaccard"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)
        union = Nu | Nv
        return float(len(Nu & Nv) / len(union)) if union else 0.0


class AdamicAdarFeature(PairFeatureStrategy):
    name = "adamic_adar"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)

        score = 0.0

        for z in Nu & Nv:
            dz = context.degree[z]
            if dz > 1:
                score += 1.0 / math.log(dz)

        return score


class ResourceAllocationFeature(PairFeatureStrategy):
    name = "resource_allocation"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)

        return float(
            sum(
                1.0 / context.degree[z]
                for z in Nu & Nv
                if context.degree[z] > 0
            )
        )


class PreferentialAttachmentFeature(PairFeatureStrategy):
    name = "preferential_attachment"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)
        return float(len(Nu) * len(Nv))


class LogDegreeSumFeature(PairFeatureStrategy):
    name = "log_degree_sum"

    def compute(self, context, u, v) -> float:
        Nu, Nv = context.neighbour_pair(u, v)
        return float(
            math.log1p(len(Nu))
            + math.log1p(len(Nv))
        )


class SameDepartmentFeature(PairFeatureStrategy):
    name = "same_department"

    def compute(self, context, u, v) -> float:
        return float(
            context.department_map[u]
            == context.department_map[v]
        )


class ReciprocityFeature(PairFeatureStrategy):
    name = "reciprocal"

    def compute(self, context, u, v) -> float:
        if context.directed is None:
            return 0.0

        return float(
            context.directed.has_edge(v, u)
        )


class CompositePairFeatureExtractor:
    def __init__(
        self,
        context: PairContext,
        strategies: Sequence[PairFeatureStrategy],
    ):
        self.context = context
        self.strategies = list(strategies)

    @property
    def feature_names(self) -> list[str]:
        return [
            strategy.name
            for strategy in self.strategies
        ]

    def one(self, u: int, v: int) -> dict[str, float]:
        return {
            strategy.name: strategy.compute(
                self.context,
                u,
                v,
            )
            for strategy in self.strategies
        }

    def transform(
        self,
        pairs: Iterable[tuple[int, int]],
    ) -> pd.DataFrame:
        return pd.DataFrame(
            [self.one(u, v) for u, v in pairs]
        )

### Why the Composite pattern is useful here

Suppose we want to add a Katz-style feature later.

We create one new strategy:

```python
class KatzFeature(PairFeatureStrategy):
    ...
```

and inject it into the composite list.

We do **not** modify:

- the sampler,
- the model tuner,
- the evaluation code,
- the existing features.

That is the **Open/Closed Principle** in a practical notebook-sized example.

## 13. Reusable dyad sampler

Positive/negative dyad sampling was another repeated task.

`DyadSampler` now handles both directed and undirected cases.

The sampler only creates pairs and labels; feature extraction is delegated elsewhere.

In [22]:
class DyadSampler:
    def __init__(
        self,
        graph: nx.Graph | nx.DiGraph,
        random_state: int = 42,
    ):
        self.graph = graph
        self.nodes = np.array(sorted(graph.nodes()))
        self.rng = np.random.default_rng(random_state)
        self.directed = graph.is_directed()

    def canonical(
        self,
        u: int,
        v: int,
    ) -> tuple[int, int]:
        if self.directed:
            return int(u), int(v)
        return (
            (int(u), int(v))
            if u < v
            else (int(v), int(u))
        )

    def sample_negative_pairs(
        self,
        n: int,
        forbidden_graph: Optional[nx.Graph | nx.DiGraph] = None,
    ) -> list[tuple[int, int]]:
        forbidden = forbidden_graph or self.graph
        negatives: set[tuple[int, int]] = set()

        while len(negatives) < n:
            u, v = self.rng.choice(
                self.nodes,
                size=2,
                replace=False,
            )

            pair = self.canonical(int(u), int(v))

            if not forbidden.has_edge(*pair):
                negatives.add(pair)

        return sorted(negatives)

    def balanced_sample(
        self,
        max_positive: int,
        negative_ratio: float = 1.0,
    ) -> tuple[list[tuple[int, int]], np.ndarray]:
        positives_all = [
            self.canonical(u, v)
            for u, v in self.graph.edges()
        ]

        n_positive = min(
            max_positive,
            len(positives_all),
        )

        chosen = self.rng.choice(
            len(positives_all),
            size=n_positive,
            replace=False,
        )

        positives = [
            positives_all[i]
            for i in chosen
        ]

        n_negative = int(round(n_positive * negative_ratio))

        negatives = self.sample_negative_pairs(
            n_negative
        )

        pairs = positives + negatives

        y = np.concatenate(
            [
                np.ones(len(positives), dtype=int),
                np.zeros(len(negatives), dtype=int),
            ]
        )

        order = self.rng.permutation(len(pairs))

        return (
            [pairs[i] for i in order],
            y[order],
        )

# Part VIII — Generic model-search framework

## 14. Factory/Registry pattern

Hyperparameter-search code is often highly repetitive.

Each model really needs only:

- an estimator,
- a search space,
- a search type,
- a number of random iterations if applicable.

`ModelSpec` stores that configuration.

`ModelSearchFactory` converts it into an actual `GridSearchCV` or `RandomizedSearchCV`.

Adding another model now means registering a new specification rather than copying a full search block.

In [23]:
def _sklearn_major_minor() -> tuple[int, int]:
    parts = sklearn.__version__.split(".")
    return int(parts[0]), int(parts[1])


def logistic_regularization_grid(
    prefix: str = "model__",
) -> dict[str, list[Any]]:
    """Return an L1/L2 tuning grid compatible with old and new sklearn APIs."""
    grid: dict[str, list[Any]] = {
        f"{prefix}C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10],
    }

    if _sklearn_major_minor() >= (1, 8):
        # sklearn >= 1.8 deprecates `penalty`; l1_ratio=0 -> L2, 1 -> L1.
        grid[f"{prefix}l1_ratio"] = [0.0, 1.0]
    else:
        grid[f"{prefix}penalty"] = ["l1", "l2"]

    return grid


@dataclass(frozen=True)
class ModelSpec:
    name: str
    estimator: BaseEstimator
    search_space: Mapping[str, Sequence[Any]]
    search_kind: str = "grid"
    n_iter: int = 20


class ModelSearchFactory:
    def __init__(
        self,
        cv,
        scoring: str = "average_precision",
        random_state: int = 42,
    ):
        self.cv = cv
        self.scoring = scoring
        self.random_state = random_state

    def build(self, spec: ModelSpec):
        common = dict(
            estimator=spec.estimator,
            scoring=self.scoring,
            cv=self.cv,
            n_jobs=-1,
            return_train_score=True,
        )

        if spec.search_kind == "grid":
            return GridSearchCV(
                param_grid=spec.search_space,
                **common,
            )

        if spec.search_kind == "random":
            return RandomizedSearchCV(
                param_distributions=spec.search_space,
                n_iter=spec.n_iter,
                random_state=self.random_state,
                **common,
            )

        raise ValueError(
            f"Unknown search_kind={spec.search_kind!r}"
        )


@dataclass
class SearchResult:
    name: str
    search: Any

    @property
    def best_estimator(self):
        return self.search.best_estimator_

    @property
    def best_score(self) -> float:
        return float(self.search.best_score_)

    @property
    def best_params(self) -> dict[str, Any]:
        return dict(self.search.best_params_)

    def cv_table(self, top_n: int = 12) -> pd.DataFrame:
        frame = pd.DataFrame(self.search.cv_results_)

        columns = [
            c
            for c in frame.columns
            if c.startswith("param_")
        ]

        columns += [
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "mean_fit_time",
            "rank_test_score",
        ]

        return (
            frame[columns]
            .sort_values("rank_test_score")
            .head(top_n)
            .reset_index(drop=True)
        )


class ModelExperiment:
    def __init__(
        self,
        specs: Sequence[ModelSpec],
        factory: ModelSearchFactory,
    ):
        self.specs = list(specs)
        self.factory = factory

    def fit_all(
        self,
        X: pd.DataFrame,
        y: np.ndarray | pd.Series,
    ) -> dict[str, SearchResult]:
        results = {}

        for spec in self.specs:
            print(f"\n{'=' * 80}")
            print(f"Tuning: {spec.name}")

            search = self.factory.build(spec)
            search.fit(X, y)

            result = SearchResult(
                name=spec.name,
                search=search,
            )

            results[spec.name] = result

            print(
                f"Best CV score: {result.best_score:.4f}"
            )
            print("Best parameters:")
            print(result.best_params)

        return results

# Part IX — ERGM-inspired conditional modelling

## 15. One shared feature system, a different feature subset

A full ERGM has

$$
P_\theta(Y=y)
=
\frac{\exp\{\theta^\top s(y)\}}{\kappa(\theta)}.
$$

For teaching, we use a logistic pseudolikelihood approximation.

The key software-design improvement is that ERGM-style modelling now reuses the **same pair-feature infrastructure** as link prediction.

For a directed dyad \(u\to v\), we include:

- reciprocity,
- same department,
- common neighbours,
- Jaccard,
- Adamic–Adar,
- degree effect.

The model is

$$
\operatorname{logit}P(Y_{uv}=1\mid Y_{-uv})
=
\beta^\top x_{uv}.
$$

In [24]:
directed_context = PairContext(
    undirected=Gu,
    directed=Gd,
    department_map=dept_map,
)

ergm_extractor = CompositePairFeatureExtractor(
    directed_context,
    strategies=[
        ReciprocityFeature(),
        SameDepartmentFeature(),
        CommonNeighbors(),
        JaccardFeature(),
        AdamicAdarFeature(),
        LogDegreeSumFeature(),
    ],
)

ergm_pairs, y_ergm = DyadSampler(
    Gd,
    random_state=CFG.random_state,
).balanced_sample(
    max_positive=CFG.ergm_positive,
    negative_ratio=1.0,
)

X_ergm = ergm_extractor.transform(ergm_pairs)

display(X_ergm.head())

display(
    X_ergm.assign(target=y_ergm)
    .groupby("target")
    .mean()
    .round(4)
)

,reciprocal,same_department,common_neighbors,jaccard,adamic_adar,log_degree_sum
0,1.0,0.0,9.0,0.045918,2.120265,8.630522
1,0.0,0.0,0.0,0.000000,0.000000,6.159095
2,0.0,0.0,0.0,0.000000,0.000000,5.780744
3,0.0,0.0,0.0,0.000000,0.000000,5.123964
4,0.0,0.0,2.0,0.015267,0.424045,7.603898


,reciprocal,same_department,common_neighbors,jaccard,adamic_adar,log_degree_sum
target,,,,,,
0,0.0075,0.0378,1.9308,0.0216,0.4207,5.6935
1,0.7088,0.3442,20.7412,0.1778,4.9163,8.0437


In [25]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=CFG.random_state,
)

ergm_spec = ModelSpec(
    name="ERGM-style Logistic Pseudolikelihood",
    estimator=Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    solver="liblinear",
                    max_iter=2_000,
                    random_state=CFG.random_state,
                ),
            ),
        ]
    ),
    search_space=logistic_regularization_grid(),
    search_kind="grid",
)

ergm_experiment = ModelExperiment(
    specs=[ergm_spec],
    factory=ModelSearchFactory(
        cv=cv,
        scoring="average_precision",
        random_state=CFG.random_state,
    ),
)

ergm_results = ergm_experiment.fit_all(
    X_ergm,
    y_ergm,
)

ergm_result = ergm_results[ergm_spec.name]

display(
    ergm_result
    .cv_table()
    .round(4)
)


Tuning: ERGM-style Logistic Pseudolikelihood
Best CV score: 0.9766
Best parameters:
{'model__C': 10, 'model__penalty': 'l1'}


,param_model__C,param_model__penalty,mean_test_score,std_test_score,mean_train_score,mean_fit_time,rank_test_score
0,10.00,l1,0.9766,0.0037,0.9769,0.2822,1
1,5.00,l1,0.9766,0.0037,0.9769,0.3197,2
2,10.00,l2,0.9765,0.0037,0.9768,0.0478,3
3,1.00,l1,0.9764,0.0037,0.9768,0.3112,4
4,5.00,l2,0.9764,0.0037,0.9767,0.0535,5
5,0.50,l1,0.9761,0.0037,0.9764,0.2697,6
6,1.00,l2,0.9758,0.0038,0.9761,0.0460,7
7,0.50,l2,0.9754,0.0039,0.9759,0.0328,8
8,0.10,l1,0.9751,0.0040,0.9755,0.0463,9
9,0.10,l2,0.9750,0.0040,0.9753,0.0321,10


In [26]:
best_ergm = ergm_result.best_estimator

ergm_coefficients = pd.DataFrame(
    {
        "feature": X_ergm.columns,
        "scaled_coefficient": best_ergm.named_steps["model"].coef_[0],
    }
)

ergm_coefficients["odds_multiplier_per_1SD"] = np.exp(
    ergm_coefficients["scaled_coefficient"]
)

ergm_coefficients["abs_coefficient"] = (
    ergm_coefficients["scaled_coefficient"].abs()
)

display(
    ergm_coefficients
    .sort_values("abs_coefficient", ascending=False)
    .drop(columns="abs_coefficient")
    .round(4)
)

,feature,scaled_coefficient,odds_multiplier_per_1SD
4,adamic_adar,10.0338,22784.5882
2,common_neighbors,-8.5036,0.0002
0,reciprocal,2.0532,7.7927
3,jaccard,1.0587,2.8826
5,log_degree_sum,1.0503,2.8586
1,same_department,0.7674,2.1541


### Interpretation

For a standardised predictor with coefficient \(\beta_j\),

$$
e^{\beta_j}
$$

is the multiplicative change in conditional odds for an approximately one-standard-deviation increase, holding the other included terms fixed.

This is **associational**, not automatically causal.

It is also an educational pseudolikelihood, not MCMC-MLE for a full ERGM.

# Part X — Link prediction

## 16. Connectivity-preserving edge holdout

The test splitter protects a spanning tree before removing test edges.

That ensures the training graph remains connected.

More importantly, **all pair features are built from the training graph**, preventing held-out edges from leaking into:

- common-neighbour counts,
- degree,
- Jaccard,
- Adamic–Adar.

In [27]:
@dataclass
class LinkSplit:
    train_graph: nx.Graph
    train_positive: list[tuple[int, int]]
    test_positive: list[tuple[int, int]]
    test_negative: list[tuple[int, int]]


class ConnectivityPreservingLinkSplitter:
    def __init__(
        self,
        graph: nx.Graph,
        random_state: int = 42,
    ):
        if not nx.is_connected(graph):
            raise ValueError(
                "Expected a connected graph."
            )

        self.graph = graph
        self.rng = np.random.default_rng(random_state)

    @staticmethod
    def canonical(
        u: int,
        v: int,
    ) -> tuple[int, int]:
        return (
            (int(u), int(v))
            if u < v
            else (int(v), int(u))
        )

    def split(
        self,
        test_fraction: float,
    ) -> LinkSplit:
        spanning_tree = nx.minimum_spanning_tree(self.graph)

        protected = {
            self.canonical(u, v)
            for u, v in spanning_tree.edges()
        }

        candidates = [
            self.canonical(u, v)
            for u, v in self.graph.edges()
            if self.canonical(u, v) not in protected
        ]

        n_test = min(
            int(round(test_fraction * self.graph.number_of_edges())),
            len(candidates),
        )

        indices = self.rng.choice(
            len(candidates),
            size=n_test,
            replace=False,
        )

        test_positive = [
            candidates[i]
            for i in indices
        ]

        train = self.graph.copy()
        train.remove_edges_from(test_positive)

        if not nx.is_connected(train):
            raise RuntimeError(
                "Connectivity protection failed unexpectedly."
            )

        negative_sampler = DyadSampler(
            self.graph,
            random_state=CFG.random_state,
        )

        test_negative = negative_sampler.sample_negative_pairs(
            n_test,
            forbidden_graph=self.graph,
        )

        train_positive = [
            self.canonical(u, v)
            for u, v in train.edges()
        ]

        return LinkSplit(
            train_graph=train,
            train_positive=train_positive,
            test_positive=test_positive,
            test_negative=test_negative,
        )


link_split = ConnectivityPreservingLinkSplitter(
    G_lcc,
    random_state=CFG.random_state,
).split(
    CFG.link_test_fraction
)

print("Training graph connected:", nx.is_connected(link_split.train_graph))
print("Held-out positives:", len(link_split.test_positive))
print("Held-out negatives:", len(link_split.test_negative))

Training graph connected: True
Held-out positives: 2410
Held-out negatives: 2410


## 17. Reuse the Composite pair-feature system

The link-prediction experiment injects a different feature set into the same composite extractor.

This is the payoff from modularisation:

```python
CompositePairFeatureExtractor(
    context,
    [
        CommonNeighbors(),
        JaccardFeature(),
        AdamicAdarFeature(),
        ...
    ],
)
```

No new feature-builder class is required.

In [28]:
link_context = PairContext(
    undirected=link_split.train_graph,
    department_map=dept_map,
)

link_extractor = CompositePairFeatureExtractor(
    link_context,
    strategies=[
        CommonNeighbors(),
        JaccardFeature(),
        AdamicAdarFeature(),
        ResourceAllocationFeature(),
        PreferentialAttachmentFeature(),
        LogDegreeSumFeature(),
        SameDepartmentFeature(),
    ],
)

train_sampler = DyadSampler(
    link_split.train_graph,
    random_state=CFG.random_state,
)

# We want training negatives that are absent from the ORIGINAL graph,
# not merely absent because they were held out.
train_positive_all = link_split.train_positive

local_rng = np.random.default_rng(CFG.random_state)

n_train_pos = min(
    CFG.link_train_positive,
    len(train_positive_all),
)

chosen = local_rng.choice(
    len(train_positive_all),
    size=n_train_pos,
    replace=False,
)

train_positive = [
    train_positive_all[i]
    for i in chosen
]

negative_sampler = DyadSampler(
    G_lcc,
    random_state=CFG.random_state + 1,
)

train_negative = negative_sampler.sample_negative_pairs(
    int(round(n_train_pos * CFG.negative_ratio)),
    forbidden_graph=G_lcc,
)

train_pairs = train_positive + train_negative

y_train = np.concatenate(
    [
        np.ones(len(train_positive), dtype=int),
        np.zeros(len(train_negative), dtype=int),
    ]
)

order = local_rng.permutation(len(train_pairs))
train_pairs = [train_pairs[i] for i in order]
y_train = y_train[order]

test_pairs = (
    link_split.test_positive
    + link_split.test_negative
)

y_test = np.concatenate(
    [
        np.ones(len(link_split.test_positive), dtype=int),
        np.zeros(len(link_split.test_negative), dtype=int),
    ]
)

X_train = link_extractor.transform(train_pairs)
X_test = link_extractor.transform(test_pairs)

print("Training matrix:", X_train.shape)
print("Test matrix:", X_test.shape)
display(X_train.describe().T.round(4))

Training matrix: (10000, 7)
Test matrix: (4820, 7)


,count,mean,std,min,25%,50%,75%,max
common_neighbors,10000.0,7.6598,10.5016,0.0000,0.0000,3.0000,12.0000,112.0000
jaccard,10000.0,0.0777,0.0921,0.0000,0.0000,0.0419,0.1250,0.5455
adamic_adar,10000.0,1.8733,2.5695,0.0000,0.0000,0.7267,2.9621,29.8505
resource_allocation,10000.0,0.1465,0.2160,0.0000,0.0000,0.0443,0.2273,2.9495
preferential_attachment,10000.0,2203.2121,3529.4129,0.0000,168.0000,812.0000,2641.2500,50760.0000
log_degree_sum,10000.0,6.5566,1.8125,1.0986,5.3471,6.7845,7.9213,10.8439
same_department,10000.0,0.1869,0.3899,0.0000,0.0000,0.0000,0.0000,1.0000


## 18. Generic score evaluator for heuristic baselines

Before training ML models, individual structural features should be tested as ranking scores.

This avoids the common mistake of comparing a complicated classifier only against random guessing.

In [29]:
class ScoreEvaluator:
    @staticmethod
    def binary_ranking(
        y_true: np.ndarray,
        scores: np.ndarray,
    ) -> dict[str, float]:
        return {
            "ROC_AUC": roc_auc_score(y_true, scores),
            "PR_AUC": average_precision_score(y_true, scores),
        }

    @staticmethod
    def precision_at_k(
        y_true: np.ndarray,
        scores: np.ndarray,
        k: int,
    ) -> float:
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1][:k]
        return float(np.mean(y_true[order]))


heuristic_names = [
    "common_neighbors",
    "jaccard",
    "adamic_adar",
    "resource_allocation",
    "preferential_attachment",
]

heuristic_results = pd.DataFrame(
    [
        {
            "heuristic": name,
            **ScoreEvaluator.binary_ranking(
                y_test,
                X_test[name].to_numpy(),
            ),
        }
        for name in heuristic_names
    ]
).sort_values("PR_AUC", ascending=False)

display(heuristic_results.round(4))

,heuristic,ROC_AUC,PR_AUC
3,resource_allocation,0.9538,0.9524
2,adamic_adar,0.9503,0.9469
0,common_neighbors,0.9457,0.9364
1,jaccard,0.9349,0.9288
4,preferential_attachment,0.8716,0.8669


## 19. Register model families

Three model families are declared as `ModelSpec` objects.

This is much less repetitive than writing three separate search pipelines.

### Logistic Regression

Interpretable linear baseline.

### Random Forest

Nonlinear interactions and threshold effects.

### HistGradientBoosting

Efficient boosting for tabular pair features.

The common CV objective is **average precision**.

In [30]:
model_specs = [
    ModelSpec(
        name="Logistic Regression",
        estimator=Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        solver="liblinear",
                        max_iter=2_000,
                        random_state=CFG.random_state,
                    ),
                ),
            ]
        ),
        search_space=logistic_regularization_grid(),
        search_kind="grid",
    ),
    ModelSpec(
        name="Random Forest",
        estimator=RandomForestClassifier(
            random_state=CFG.random_state,
            n_jobs=-1,
        ),
        search_space={
            "n_estimators": [150, 250, 400, 600],
            "max_depth": [None, 6, 10, 16, 24],
            "min_samples_leaf": [1, 2, 5, 10, 20],
            "max_features": ["sqrt", "log2", 0.5, 0.8],
            "criterion": ["gini", "entropy"],
        },
        search_kind="random",
        n_iter=12 if CFG.fast_mode else 30,
    ),
    ModelSpec(
        name="HistGradientBoosting",
        estimator=HistGradientBoostingClassifier(
            random_state=CFG.random_state,
        ),
        search_space={
            "learning_rate": [0.02, 0.04, 0.07, 0.10, 0.15, 0.20],
            "max_iter": [100, 200, 350, 500],
            "max_leaf_nodes": [7, 15, 31, 63],
            "min_samples_leaf": [5, 10, 20, 40, 80],
            "l2_regularization": [0.0, 0.1, 0.5, 1.0, 5.0, 10.0],
        },
        search_kind="random",
        n_iter=12 if CFG.fast_mode else 35,
    ),
]

link_experiment = ModelExperiment(
    specs=model_specs,
    factory=ModelSearchFactory(
        cv=cv,
        scoring="average_precision",
        random_state=CFG.random_state,
    ),
)

link_search_results = link_experiment.fit_all(
    X_train,
    y_train,
)


Tuning: Logistic Regression
Best CV score: 0.9543
Best parameters:
{'model__C': 10, 'model__penalty': 'l1'}

Tuning: Random Forest
Best CV score: 0.9582
Best parameters:
{'n_estimators': 400, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 6, 'criterion': 'entropy'}

Tuning: HistGradientBoosting
Best CV score: 0.9581
Best parameters:
{'min_samples_leaf': 20, 'max_leaf_nodes': 7, 'max_iter': 350, 'learning_rate': 0.02, 'l2_regularization': 1.0}


## 20. Generic hold-out evaluator

Evaluation is another natural place to centralise repeated logic.

`BinaryModelEvaluator` accepts any fitted classifier implementing `predict_proba`.

In [31]:
class BinaryModelEvaluator:
    def __init__(
        self,
        X_test: pd.DataFrame,
        y_test: np.ndarray,
    ):
        self.X_test = X_test
        self.y_test = y_test

    def evaluate(
        self,
        name: str,
        estimator,
        threshold: float = 0.5,
    ) -> dict[str, float | str]:
        probability = estimator.predict_proba(
            self.X_test
        )[:, 1]

        prediction = (
            probability >= threshold
        ).astype(int)

        return {
            "model": name,
            "ROC_AUC": roc_auc_score(
                self.y_test,
                probability,
            ),
            "PR_AUC": average_precision_score(
                self.y_test,
                probability,
            ),
            "precision@0.5": precision_score(
                self.y_test,
                prediction,
            ),
            "recall@0.5": recall_score(
                self.y_test,
                prediction,
            ),
            "F1@0.5": f1_score(
                self.y_test,
                prediction,
            ),
        }

    def probability_map(
        self,
        results: Mapping[str, SearchResult],
    ) -> dict[str, np.ndarray]:
        return {
            name: result.best_estimator.predict_proba(
                self.X_test
            )[:, 1]
            for name, result in results.items()
        }


evaluator = BinaryModelEvaluator(
    X_test,
    y_test,
)

model_results = pd.DataFrame(
    [
        evaluator.evaluate(
            name,
            result.best_estimator,
        )
        for name, result in link_search_results.items()
    ]
).sort_values("PR_AUC", ascending=False)

display(model_results.round(4))

,model,ROC_AUC,PR_AUC,precision@0.5,recall@0.5,F1@0.5
2,HistGradientBoosting,0.9584,0.9556,0.8959,0.8967,0.8963
1,Random Forest,0.9588,0.9555,0.9021,0.8950,0.8986
0,Logistic Regression,0.9572,0.9554,0.9177,0.8647,0.8904


## 21. ROC and Precision–Recall curves

One curve builder now works for every fitted model.

In [32]:
class CurveFactory:
    @staticmethod
    def roc_frame(
        y_true: np.ndarray,
        probabilities: Mapping[str, np.ndarray],
    ) -> pd.DataFrame:
        rows = []

        for name, prob in probabilities.items():
            fpr, tpr, _ = roc_curve(y_true, prob)

            rows.extend(
                {
                    "model": name,
                    "x": x,
                    "y": y,
                }
                for x, y in zip(fpr, tpr)
            )

        return pd.DataFrame(rows)

    @staticmethod
    def pr_frame(
        y_true: np.ndarray,
        probabilities: Mapping[str, np.ndarray],
    ) -> pd.DataFrame:
        rows = []

        for name, prob in probabilities.items():
            precision, recall, _ = precision_recall_curve(
                y_true,
                prob,
            )

            rows.extend(
                {
                    "model": name,
                    "x": r,
                    "y": p,
                }
                for p, r in zip(precision, recall)
            )

        return pd.DataFrame(rows)

    @staticmethod
    def plot_curves(
        frame: pd.DataFrame,
        title: str,
        x_label: str,
        y_label: str,
        legend_location: str = "bottom_right",
    ):
        p = figure(
            width=780,
            height=450,
            title=title,
            x_axis_label=x_label,
            y_axis_label=y_label,
        )

        for name, group in frame.groupby("model"):
            p.line(
                group["x"],
                group["y"],
                line_width=3,
                legend_label=name,
            )

        if len(p.legend):
            p.legend.location = legend_location
            p.legend.click_policy = "hide"

        return p


probabilities = evaluator.probability_map(
    link_search_results
)

roc_frame = CurveFactory.roc_frame(
    y_test,
    probabilities,
)

pr_frame = CurveFactory.pr_frame(
    y_test,
    probabilities,
)

show(
    CurveFactory.plot_curves(
        roc_frame,
        title="ROC curves — tuned link-prediction models",
        x_label="False Positive Rate",
        y_label="True Positive Rate",
    )
)

show(
    CurveFactory.plot_curves(
        pr_frame,
        title="Precision–Recall curves — tuned link-prediction models",
        x_label="Recall",
        y_label="Precision",
    )
)

## 22. Precision@K with one generic evaluator

Many network applications return a shortlist rather than classify every possible pair.

$$
\text{Precision@K}
=
\frac{\text{true links in top K}}{K}.
$$

In [33]:
k_values = [25, 50, 100, 250, 500]

ranking_rows = []

for model_name, scores in probabilities.items():
    for k in k_values:
        ranking_rows.append(
            {
                "model": model_name,
                "K": k,
                "Precision@K": ScoreEvaluator.precision_at_k(
                    y_test,
                    scores,
                    k,
                ),
            }
        )

ranking_df = pd.DataFrame(ranking_rows)

display(
    ranking_df
    .pivot(
        index="K",
        columns="model",
        values="Precision@K",
    )
    .round(4)
)

model,HistGradientBoosting,Logistic Regression,Random Forest
K,,,
25,1.000,1.000,0.960
50,0.980,1.000,0.980
100,0.990,1.000,0.990
250,0.996,0.992,0.996
500,0.990,0.988,0.988


## 23. Inspect tuned search surfaces and model interpretation

Because every search is wrapped in `SearchResult`, inspecting top CV configurations is now identical across model families.

In [34]:
for name, result in link_search_results.items():
    print(f"\n{name}")
    display(
        result
        .cv_table(top_n=10)
        .round(4)
    )


Logistic Regression


,param_model__C,param_model__penalty,mean_test_score,std_test_score,mean_train_score,mean_fit_time,rank_test_score
0,10.0,l1,0.9543,0.0060,0.9545,1.7456,1
1,5.0,l1,0.9542,0.0060,0.9544,1.4296,2
2,10.0,l2,0.9542,0.0060,0.9543,0.0677,3
3,5.0,l2,0.9541,0.0060,0.9542,0.0785,4
4,1.0,l1,0.9540,0.0061,0.9541,0.1064,5
5,1.0,l2,0.9540,0.0061,0.9541,0.0537,6
6,0.5,l1,0.9539,0.0061,0.9541,0.1023,7
7,0.5,l2,0.9539,0.0060,0.9541,0.0456,8
8,0.1,l2,0.9536,0.0060,0.9537,0.0460,9
9,0.1,l1,0.9536,0.0060,0.9537,0.0551,10



Random Forest


,param_n_estimators,param_min_samples_leaf,param_max_features,param_max_depth,param_criterion,mean_test_score,std_test_score,mean_train_score,mean_fit_time,rank_test_score
0,400,5,sqrt,6,entropy,0.9582,0.0047,0.9667,10.3842,1
1,250,10,0.5,6,entropy,0.9580,0.0049,0.9668,8.7415,2
2,150,20,0.5,16,entropy,0.9558,0.0057,0.9728,3.6111,3
3,150,20,0.5,None,entropy,0.9558,0.0055,0.9729,4.5816,4
4,250,1,0.8,10,entropy,0.9557,0.0047,0.9868,13.3721,5
5,600,2,log2,16,entropy,0.9527,0.0046,0.9947,19.2171,6
6,400,2,0.5,16,gini,0.9519,0.0048,0.9957,11.7559,7
7,400,2,log2,24,gini,0.9519,0.0053,0.9966,12.6044,8
8,400,2,0.8,None,gini,0.9502,0.0052,0.9979,19.6039,9
9,250,2,0.8,None,gini,0.9501,0.0053,0.9978,11.1181,10



HistGradientBoosting


,param_min_samples_leaf,param_max_leaf_nodes,param_max_iter,param_learning_rate,param_l2_regularization,mean_test_score,std_test_score,mean_train_score,mean_fit_time,rank_test_score
0,20,7,350,0.02,1.0,0.9581,0.0050,0.9633,1.6080,1
1,5,31,100,0.07,0.5,0.9569,0.0049,0.9803,1.0579,2
2,80,15,100,0.10,5.0,0.9565,0.0055,0.9695,0.7157,3
3,5,31,350,0.04,0.0,0.9544,0.0050,0.9905,3.1244,4
4,5,31,100,0.15,0.0,0.9538,0.0050,0.9915,1.0934,5
5,80,31,100,0.15,0.5,0.9533,0.0062,0.9822,1.2666,6
6,5,7,500,0.15,0.1,0.9524,0.0042,0.9886,1.9256,7
7,5,63,350,0.04,0.5,0.9522,0.0056,0.9951,7.0640,8
8,40,63,200,0.07,1.0,0.9501,0.0067,0.9905,3.2774,9
9,5,63,350,0.07,5.0,0.9496,0.0069,0.9936,5.5262,10


In [35]:
logistic_result = link_search_results[
    "Logistic Regression"
]

logistic_model = logistic_result.best_estimator

logistic_coefficients = pd.DataFrame(
    {
        "feature": X_train.columns,
        "coefficient": logistic_model.named_steps["model"].coef_[0],
    }
)

logistic_coefficients["abs_coefficient"] = (
    logistic_coefficients["coefficient"].abs()
)

display(
    logistic_coefficients
    .sort_values("abs_coefficient", ascending=False)
    .drop(columns="abs_coefficient")
    .round(4)
)


rf_result = link_search_results[
    "Random Forest"
]

rf_importance = pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": rf_result.best_estimator.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(rf_importance.round(4))

,feature,coefficient
2,adamic_adar,9.7027
0,common_neighbors,-6.2449
5,log_degree_sum,1.2553
6,same_department,0.9629
4,preferential_attachment,-0.8014
1,jaccard,0.4667
3,resource_allocation,-0.2363


,feature,importance
3,resource_allocation,0.2914
2,adamic_adar,0.2492
0,common_neighbors,0.1804
1,jaccard,0.1402
4,preferential_attachment,0.0577
5,log_degree_sum,0.0486
6,same_department,0.0325


# Part XI — Facade: `NetworkWorkbench`

The notebook now has several reusable components.

A **Facade** provides a small entry point for common tasks without hiding the underlying classes.

This is useful in future notebooks:

```python
workbench = NetworkWorkbench(dataset)
workbench.summary()
workbench.centrality()
workbench.spectral()
```

The facade is convenience only; it does not contain the actual statistical logic.

In [36]:
class NetworkWorkbench:
    def __init__(self, dataset: NetworkDataset):
        self.dataset = dataset
        self.analyzer = NetworkAnalyzer(dataset)

    def summary(self) -> pd.DataFrame:
        return self.analyzer.global_summary()

    def degrees(self) -> pd.DataFrame:
        return self.analyzer.degree_table()

    def centrality(self) -> pd.DataFrame:
        return self.analyzer.centrality_table()

    def spectral(self) -> SpectralAnalyzer:
        return SpectralAnalyzer(
            self.dataset.largest_connected_undirected
        )

    def null_experiment(
        self,
        strategies: Optional[
            Sequence[NullModelStrategy]
        ] = None,
    ) -> NullModelExperiment:
        return NullModelExperiment(
            self.dataset.largest_connected_undirected,
            strategies=(
                list(strategies)
                if strategies is not None
                else [
                    ErdosRenyiStrategy(),
                    DegreePreservingStrategy(),
                ]
            ),
            random_state=CFG.random_state,
        )

    def community_experiment(self) -> CommunityExperiment:
        return CommunityExperiment(
            self.dataset.largest_connected_undirected,
            ground_truth=self.dataset.department_map,
            random_state=CFG.random_state,
        )


workbench = NetworkWorkbench(dataset)

display(workbench.summary().round(5))

,metric,value
0,nodes,1005.00000
1,directed_edges,24929.00000
2,undirected_edges,16064.00000
3,directed_density,0.02471
4,undirected_density,0.03184
5,reciprocity,0.71122
6,weak_components,20.00000
7,strong_components,203.00000
8,largest_WCC_nodes,986.00000
9,largest_SCC_nodes,803.00000


# Part XII — Creative extension: plugin-like feature registration

Because pair features are strategies, we can build a tiny registry.

This is useful when experimenting with dozens of network scores.

The registry maps a string name to a feature strategy class.

In [37]:
class PairFeatureRegistry:
    _registry: dict[str, type[PairFeatureStrategy]] = {}

    @classmethod
    def register(
        cls,
        feature_cls: type[PairFeatureStrategy],
    ) -> type[PairFeatureStrategy]:
        cls._registry[feature_cls.name] = feature_cls
        return feature_cls

    @classmethod
    def create(
        cls,
        names: Sequence[str],
    ) -> list[PairFeatureStrategy]:
        unknown = [
            name
            for name in names
            if name not in cls._registry
        ]

        if unknown:
            raise KeyError(
                f"Unknown pair features: {unknown}"
            )

        return [
            cls._registry[name]()
            for name in names
        ]

    @classmethod
    def available(cls) -> list[str]:
        return sorted(cls._registry)


for feature_cls in [
    CommonNeighbors,
    JaccardFeature,
    AdamicAdarFeature,
    ResourceAllocationFeature,
    PreferentialAttachmentFeature,
    LogDegreeSumFeature,
    SameDepartmentFeature,
    ReciprocityFeature,
]:
    PairFeatureRegistry.register(feature_cls)


print(
    "Available registered features:",
    PairFeatureRegistry.available(),
)

Available registered features: ['adamic_adar', 'common_neighbors', 'jaccard', 'log_degree_sum', 'preferential_attachment', 'reciprocal', 'resource_allocation', 'same_department']


### Example: configure features declaratively

Instead of editing feature-building code:

```python
selected = [
    "common_neighbors",
    "adamic_adar",
    "same_department",
]
```

The registry constructs the strategies.

This is a lightweight **plugin architecture**.

In [38]:
selected_feature_names = [
    "common_neighbors",
    "jaccard",
    "adamic_adar",
    "same_department",
]

declarative_extractor = CompositePairFeatureExtractor(
    context=link_context,
    strategies=PairFeatureRegistry.create(
        selected_feature_names
    ),
)

display(
    declarative_extractor
    .transform(test_pairs[:8])
    .round(4)
)

,common_neighbors,jaccard,adamic_adar,same_department
0,0.0,0.0000,0.0000,0.0
1,2.0,0.0571,0.5783,0.0
2,13.0,0.3421,4.0335,1.0
3,7.0,0.1458,1.8868,0.0
4,11.0,0.1028,2.5387,1.0
5,9.0,0.0811,2.1884,0.0
6,11.0,0.0803,2.5195,0.0
7,6.0,0.1200,1.8722,1.0


# Part XIII — Feature ablation experiment

A modular feature registry makes **ablation studies** easy.

Ablation asks:

> What happens if one source of network information is removed?

Examples:

- remove department information,
- remove degree effects,
- use only common-neighbour-family features.

This is often more informative than adding ever more complicated models.

In [39]:
class FeatureAblationExperiment:
    def __init__(
        self,
        context: PairContext,
        train_pairs: Sequence[tuple[int, int]],
        y_train: np.ndarray,
        test_pairs: Sequence[tuple[int, int]],
        y_test: np.ndarray,
        random_state: int = 42,
    ):
        self.context = context
        self.train_pairs = train_pairs
        self.y_train = y_train
        self.test_pairs = test_pairs
        self.y_test = y_test
        self.random_state = random_state

    def run(
        self,
        feature_sets: Mapping[str, Sequence[str]],
    ) -> pd.DataFrame:
        rows = []

        for experiment_name, names in feature_sets.items():
            extractor = CompositePairFeatureExtractor(
                self.context,
                PairFeatureRegistry.create(names),
            )

            Xtr = extractor.transform(self.train_pairs)
            Xte = extractor.transform(self.test_pairs)

            model = Pipeline(
                [
                    ("scale", StandardScaler()),
                    (
                        "model",
                        LogisticRegression(
                            C=1.0,
                            solver="liblinear",
                            max_iter=2_000,
                            random_state=self.random_state,
                        ),
                    ),
                ]
            )

            model.fit(Xtr, self.y_train)
            prob = model.predict_proba(Xte)[:, 1]

            rows.append(
                {
                    "feature_set": experiment_name,
                    "n_features": len(names),
                    "ROC_AUC": roc_auc_score(
                        self.y_test,
                        prob,
                    ),
                    "PR_AUC": average_precision_score(
                        self.y_test,
                        prob,
                    ),
                }
            )

        return (
            pd.DataFrame(rows)
            .sort_values("PR_AUC", ascending=False)
            .reset_index(drop=True)
        )


ablation = FeatureAblationExperiment(
    context=link_context,
    train_pairs=train_pairs,
    y_train=y_train,
    test_pairs=test_pairs,
    y_test=y_test,
    random_state=CFG.random_state,
)

ablation_results = ablation.run(
    {
        "all": [
            "common_neighbors",
            "jaccard",
            "adamic_adar",
            "resource_allocation",
            "preferential_attachment",
            "log_degree_sum",
            "same_department",
        ],
        "topology_only": [
            "common_neighbors",
            "jaccard",
            "adamic_adar",
            "resource_allocation",
            "preferential_attachment",
            "log_degree_sum",
        ],
        "overlap_only": [
            "common_neighbors",
            "jaccard",
            "adamic_adar",
            "resource_allocation",
        ],
        "degree_only": [
            "preferential_attachment",
            "log_degree_sum",
        ],
        "department_only": [
            "same_department",
        ],
    }
)

display(ablation_results.round(4))

,feature_set,n_features,ROC_AUC,PR_AUC
0,all,7,0.9569,0.9550
1,topology_only,6,0.9541,0.9533
2,overlap_only,4,0.9523,0.9510
3,degree_only,2,0.8709,0.8667
4,department_only,1,0.6432,0.6282


### Why ablation is valuable

Suppose the full model performs well.

That does not tell us *why*.

Ablation can reveal whether predictive power comes mostly from:

- local topology,
- node popularity,
- department labels,
- interactions among them.

That converts a black-box performance result into a more scientific experiment.

# Part XIV — Optional temporal extension

For genuine forecasting, SNAP also provides:

- `email-Eu-core-temporal.txt.gz`

with rows

$$
(\text{source},\text{destination},\text{timestamp}).
$$

A temporal design should use:

$$
G_{\leq t_0}
\rightarrow
\text{predict new links after }t_0.
$$

This avoids pretending that a static edge list has a natural IID train/test split.

**Important:** SNAP notes that temporal node IDs do not directly correspond to the static dataset IDs, so do not naively attach the static department labels.

In [40]:
class TemporalEmailSource:
    FILE = "email-Eu-core-temporal.txt.gz"
    URL = "https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz"

    def __init__(self, data_dir: Path):
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def load(self) -> pd.DataFrame:
        path = self.data_dir / self.FILE

        if not path.exists():
            urllib.request.urlretrieve(
                self.URL,
                path,
            )

        return pd.read_csv(
            path,
            sep=r"\s+",
            header=None,
            names=["src", "dst", "timestamp"],
            compression="gzip",
            dtype={
                "src": int,
                "dst": int,
                "timestamp": np.int64,
            },
        )


if CFG.run_temporal_extension:
    temporal_df = (
        TemporalEmailSource(CFG.data_dir)
        .load()
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    cutoff = temporal_df["timestamp"].quantile(0.70)

    past = temporal_df[
        temporal_df["timestamp"] <= cutoff
    ]

    future = temporal_df[
        temporal_df["timestamp"] > cutoff
    ]

    past_dyads = set(
        map(
            tuple,
            past[["src", "dst"]]
            .drop_duplicates()
            .to_numpy(),
        )
    )

    future_dyads = set(
        map(
            tuple,
            future[["src", "dst"]]
            .drop_duplicates()
            .to_numpy(),
        )
    )

    print("Past events:", len(past))
    print("Future events:", len(future))
    print(
        "Genuinely new future dyads:",
        len(future_dyads - past_dyads),
    )
else:
    print(
        "Temporal extension disabled. "
        "Set run_temporal_extension=True in Config to execute."
    )

Temporal extension disabled. Set run_temporal_extension=True in Config to execute.


# Part XV — Complexity and design trade-offs

## 24. Algorithmic complexity

Let

$$
n=|V|,
\qquad
m=|E|.
$$

| Operation | Typical complexity |
|---|---:|
| Adjacency matrix storage | \(\Theta(n^2)\) |
| Sparse adjacency / adjacency list | \(\Theta(n+m)\) |
| BFS / DFS | \(\Theta(n+m)\) |
| Single-source unweighted shortest paths | \(\Theta(n+m)\) |
| Brandes betweenness, unweighted | \(O(nm)\) |
| Dense eigendecomposition | \(O(n^3)\) |
| Sparse PageRank iteration | \(O(m)\) per iteration |
| Enumerating every undirected dyad | \(O(n^2)\) |

### Pair-feature cache

`PairContext` stores each neighbour set once:

$$
\Theta(n+m)
$$

space overall for a sparse graph representation.

Then intersection-based features cost approximately

$$
O(\min(d_u,d_v))
$$

with hash-set intersection behaviour, rather than repeatedly rebuilding neighbourhood containers.

---

## 25. Design-pattern trade-offs

Patterns are useful only when they buy something.

### Strategy

Good when algorithms are genuinely interchangeable.

Used for:

- null models,
- community detectors,
- pair features.

### Composite

Good when many small operations should behave like one larger operation.

Used for:

- dyadic feature extraction.

### Factory/Registry

Good when the user wants to configure experiments declaratively.

Used for:

- model search,
- feature selection.

### Repository

Good when acquisition/storage should be isolated from statistical logic.

### Facade

Good for convenience, but should stay thin.

### What we intentionally did **not** do

We did not create:

- a class for every plot,
- dozens of tiny getters,
- deep inheritance trees,
- a dependency-injection container,
- abstract classes where a simple function is enough.

The goal is **low repetition + easy extension**, not maximum OOP.

# Part XVI — Experiments to extend the architecture

Because the code is now modular, several advanced experiments become relatively small additions.

## A. Add a Katz feature strategy

Implement

$$
K
=
(I-\beta A)^{-1}-I,
$$

or a truncated walk expansion

$$
K(u,v)
=
\sum_{\ell=1}^{L}
\beta^\ell(A^\ell)_{uv}.
$$

Register it with `PairFeatureRegistry` and rerun the ablation study.

---

## B. Hard-negative sampling strategy

Create:

```text
NegativeSamplingStrategy
├── UniformNonEdge
├── TwoHopNonEdge
└── SameDepartmentNonEdge
```

Then compare how link-prediction performance changes as negatives become harder.

---

## C. Community algorithm plugin

Implement another `CommunityStrategy`:

- greedy modularity,
- label propagation,
- Leiden through `igraph`,
- Infomap.

The existing `CommunityExperiment` can evaluate it without modification.

---

## D. Degree-corrected SBM

Model

$$
P(A_{ij}=1)
\propto
\theta_i\theta_jB_{z_i z_j}.
$$

Compare it with the ordinary SBM.

---

## E. Full ERGM

Use R `statnet::ergm` for terms such as:

- `edges`,
- `mutual`,
- `gwesp`,
- `nodematch("department")`.

Compare MCMC-MLE estimates with the notebook's educational pseudolikelihood.

---

## F. Node embeddings

Add an embedding feature strategy:

- DeepWalk,
- node2vec,
- GraphSAGE embeddings.

Then ask whether learned representations improve PR-AUC over handcrafted topology features.

---

## G. Temporal experiment strategy

Create a `TemporalSplitStrategy` supporting:

- single cutoff,
- expanding-window evaluation,
- rolling-window evaluation.

That would make the forecasting workflow consistent with the same experiment architecture.

# Final engineering and statistical takeaways

The refactoring changes the notebook from:

```text
cell-specific code
→ another cell-specific implementation
→ repeated feature logic
→ repeated model-search logic
```

into:

```text
DataSource
→ NetworkRepository
→ NetworkDataset
→ reusable analyzers
→ interchangeable strategies
→ generic experiments
→ common evaluators
```

The statistical story remains:

$$
\boxed{
\text{Observe}
\rightarrow
\text{Represent}
\rightarrow
\text{Measure}
\rightarrow
\text{Model}
\rightarrow
\text{Infer}
\rightarrow
\text{Predict}
}
$$

but the software architecture now mirrors the scientific workflow.

The most valuable design choices are not inheritance itself. They are:

1. **single responsibility**,
2. **dependency injection**,
3. **small interchangeable strategies**,
4. **shared experiment runners**,
5. **cached graph primitives**,
6. **declarative model/feature registration**,
7. **separation of analysis from presentation**.

That combination makes it much easier to ask new ST5225 questions without rewriting the notebook.